# Start here

Twelve notebooks. Each runs top to bottom against the real repository, calls the
same library a script or a test would, and is committed with its outputs, so every
number is readable without a GPU and a change in one shows up in a diff.

They are not documentation written beside the code. They are the code, run.

## What the project claims

PSBD detects a backdoor by perturbing a model at inference and watching how much
its confidence in its own prediction falls. Porting it from ConvNets to
transformers forces a question the original never faced: a transformer block has a
dozen places a perturbation could go, and they are not equivalent.

That splits the design into 2 axes, **where** a probe attaches and **what** it
does, and the measured result is that where dominates what.

## Read in this order

**The method, if you are new to it**

| # | notebook | question |
|---|---|---|
| 03 | attacks and triggers | what the 10 attacks actually do to an image |
| 04 | datasets and splits | what data each split holds and why they are paired |
| 05 | probe positions and operators | the 2 axes, and why the same rate is a different disturbance at each site |
| 06 | PSBD end to end | one checkpoint from raw forward passes to a detection verdict |

**What we found**

| # | notebook | question |
|---|---|---|
| 01 | latent distribution | what the trigger does to the representation, layer by layer |
| 02 | architecture and attack comparison | what generalizes across ViT, Swin and 4 attacks |
| 07 | locating the backdoor | where it lives, and whether the direction is causally sufficient |
| 08 | detection results | the whole cached result set, including its coverage holes |
| 09 | placement search | the headline, and what does and does not reconcile |
| 10 | geometry predicts detection | can latent geometry predict detection without running the detector |
| 11 | adaptive attacker and multi-probe | what happens when the attacker knows the defence |

**Whether to believe any of it**

| # | notebook | question |
|---|---|---|
| 12 | how we know the numbers are right | the 3 failure modes, and the checks that now stand in the way |
| 13 | operating points and placement | AUROC and TPR at 1, 5 and 10 percent FPR for every operator, position, attack and depth band |

If you read only 3, read **06** for the method, **13** for the numbers at
deployable operating points, and **12** for whether to believe them.

In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import glob
import json

import pandas as pd

rows = []
for path in sorted(glob.glob("notebooks/*.ipynb")):
    notebook = json.load(open(path))
    code_cells = [c for c in notebook["cells"] if c["cell_type"] == "code"]
    rows.append(
        {
            "notebook": os.path.basename(path),
            "code cells": len(code_cells),
            "figures": sum(
                1
                for c in code_cells
                for o in c.get("outputs", [])
                if "image/png" in o.get("data", {})
            ),
            "errors": sum(
                1
                for c in code_cells
                for o in c.get("outputs", [])
                if o.get("output_type") == "error"
            ),
        }
    )
pd.DataFrame(rows).set_index("notebook")

,code cells,figures,errors
notebook,,,
00-start-here.ipynb,2,0,0
01-latent-distribution.ipynb,10,5,0
02-architecture-and-attack-comparison.ipynb,6,3,0
03-attacks-and-triggers.ipynb,13,3,0
04-datasets-and-splits.ipynb,17,2,0
05-probe-positions-and-operators.ipynb,12,2,0
06-psbd-end-to-end.ipynb,11,3,0
07-locating-the-backdoor.ipynb,11,4,0
08-detection-results.ipynb,24,5,0


Zero errors is the bar. A notebook that does not run is not a record of anything.

## What is settled, and what is not

The distinction matters more here than usual, because auditing this project
withdrew 2 headline numbers and downgraded a third. What follows separates what
survived from what did not.

In [2]:
claims = pd.DataFrame(
    [
        ("token_mask @ before_attention_norm: 0.911 mean, 0.632 worst, 0 inversions",
         "verified, 48 of 48 cells", "09"),
        ("placement gain over the published ConvNet placement, matched strength",
         "+0.089 panel, +0.166 CIFAR-100 1%", "09"),
        ("position dominates operator",
         "supported, but the documented 1.43x does not reproduce (1.76 here)", "09"),
        ("PSBD's neuron-bias mechanism is wrong on ViT",
         "verified: 0.0000 of shifted clean predictions reach the target class", "06"),
        ("an operator removing no capacity competes with ones that do",
         "restated: gaussian 0.897, 6th not 1st, within 0.002 of channel_mask", "08"),
        ("the backdoor direction is causally sufficient",
         "verified: 43% of clean images flip at its natural magnitude, benign control 0.007", "07"),
        ("latent collapse predicts detection beyond attack success",
         "partial rho -0.31 holding ASR fixed, vs 0.095 for raw separability", "10"),
        ("rank collapse is the MECHANISM behind the margin",
         "DOWNGRADED: the head's null space lets rank move 0.21 to 1.00 at fixed logits", "10"),
        ("evading one probe does not evade the detector",
         "the attacker inverts the probed one; neither union beats the best single probe", "11"),
        ("gain_scale @ mlp_norm_out gains +0.258",
         "WITHDRAWN: unmatched strength, -0.007 at matched strength", "09"),
        ("perturbing every block is the right default",
         "CONTRADICTED: a depth band beats all-blocks on 8 of 9 attacks, median +0.103", "13"),
        ("AUROC describes what deployment gets",
         "no: 0.826 AUROC is 0.124 TPR at 1% FPR on the same cells", "13"),
    ],
    columns=["claim", "status", "notebook"],
)
pd.set_option("display.max_colwidth", 78)
claims

,claim,status,notebook
0,"token_mask @ before_attention_norm: 0.911 mean, 0.632 worst, 0 inversions","verified, 48 of 48 cells",09
1,"placement gain over the published ConvNet placement, matched strength","+0.089 panel, +0.166 CIFAR-100 1%",09
2,position dominates operator,"supported, but the documented 1.43x does not reproduce (1.76 here)",09
3,PSBD's neuron-bias mechanism is wrong on ViT,verified: 0.0000 of shifted clean predictions reach the target class,06
4,an operator removing no capacity competes with ones that do,"restated: gaussian 0.897, 6th not 1st, within 0.002 of channel_mask",08
5,the backdoor direction is causally sufficient,"verified: 43% of clean images flip at its natural magnitude, benign contro...",07
6,latent collapse predicts detection beyond attack success,"partial rho -0.31 holding ASR fixed, vs 0.095 for raw separability",10
7,rank collapse is the MECHANISM behind the margin,DOWNGRADED: the head's null space lets rank move 0.21 to 1.00 at fixed logits,10
8,evading one probe does not evade the detector,the attacker inverts the probed one; neither union beats the best single p...,11
9,gain_scale @ mlp_norm_out gains +0.258,"WITHDRAWN: unmatched strength, -0.007 at matched strength",09


## Known gaps, stated rather than left to be discovered

- **Every number is single seed.** The largest missing piece and the most likely
  reason a security venue would reject. Replication is running.
- **No EOT** over the dropout randomness, which Athalye et al. make mandatory for
  evaluating any defence with a stochastic component.
- **Swin has no GTSRB sweep**, so any mean over datasets averages 4 for ViT and 3
  for Swin. 40 trained checkpoints exist and were never swept; that is running.
- **The competitor baselines have never been run.** SCALE-UP, IBD-PSC and TeCo are
  implemented and sign-verified but have zero recorded numbers.
- **The novelty search covers arXiv, not venue-only publications.** A paywalled
  LNCS 2026 title sits close enough to matter and has not been obtained.

## Conventions

Every notebook opens with the same setup cell anchoring at the repository root, so
the library's default relative paths resolve exactly as they do from a script.
Plots return a Figure and nothing writes an image to disk. Tables are DataFrames
built from the plain dicts the library returns. See `notebooks/README.md`.